In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"), temperature=0.4)

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harih\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7007.53it/s]


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [4]:
import os
from langchain_community.vectorstores import Chroma

os.makedirs("./data/chroma_db", exist_ok=True)

vectorstore = Chroma(
    persist_directory="./data/chroma_db",
    embedding_function=embedding_model
)

C:\Users\harih\AppData\Local\Temp\ipykernel_12668\3714221711.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\harih\AppData\Local\Temp\ipykernel_12668\3714221711.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [5]:
from langchain_core.documents import Document

sample_text = """Artificial intelligence (AI) is the capability of computational
systems to perform tasks typically associated with human intelligence, such as
learning, reasoning, problem-solving, perception, and decision-making.

LangChain is a framework for building applications powered by large language
models. It provides abstractions for chains, agents, memory, and retrieval,
making it easier to combine LLMs with external tools and data sources.

Retrieval-Augmented Generation (RAG) is a technique where relevant documents
are retrieved from a knowledge base and passed to an LLM as context, so it can
answer questions grounded in real data rather than relying only on what it
memorized during training."""

docs = [Document(page_content=sample_text, metadata={"source": "sample_notes"})]
docs

[Document(metadata={'source': 'sample_notes'}, page_content='Artificial intelligence (AI) is the capability of computational\nsystems to perform tasks typically associated with human intelligence, such as\nlearning, reasoning, problem-solving, perception, and decision-making.\n\nLangChain is a framework for building applications powered by large language\nmodels. It provides abstractions for chains, agents, memory, and retrieval,\nmaking it easier to combine LLMs with external tools and data sources.\n\nRetrieval-Augmented Generation (RAG) is a technique where relevant documents\nare retrieved from a knowledge base and passed to an LLM as context, so it can\nanswer questions grounded in real data rather than relying only on what it\nmemorized during training.')]

In [6]:
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")
chunks[0]

Split into 1 chunks


Document(metadata={'source': 'sample_notes'}, page_content='Artificial intelligence (AI) is the capability of computational\nsystems to perform tasks typically associated with human intelligence, such as\nlearning, reasoning, problem-solving, perception, and decision-making.\n\nLangChain is a framework for building applications powered by large language\nmodels. It provides abstractions for chains, agents, memory, and retrieval,\nmaking it easier to combine LLMs with external tools and data sources.\n\nRetrieval-Augmented Generation (RAG) is a technique where relevant documents\nare retrieved from a knowledge base and passed to an LLM as context, so it can\nanswer questions grounded in real data rather than relying only on what it\nmemorized during training.')

In [7]:
vectorstore.add_documents(chunks)
print("Documents added to vectorstore.")

Documents added to vectorstore.


In [8]:
results = vectorstore.similarity_search("What is LangChain?", k=3)

for i, r in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(r.page_content)
    print()

--- Result 1 ---
Artificial intelligence (AI) is the capability of computational
systems to perform tasks typically associated with human intelligence, such as
learning, reasoning, problem-solving, perception, and decision-making.

LangChain is a framework for building applications powered by large language
models. It provides abstractions for chains, agents, memory, and retrieval,
making it easier to combine LLMs with external tools and data sources.

Retrieval-Augmented Generation (RAG) is a technique where relevant documents
are retrieved from a knowledge base and passed to an LLM as context, so it can
answer questions grounded in real data rather than relying only on what it
memorized during training.



In [9]:
def answer_from_knowledge_base(query: str, k: int = 3) -> str:
    context = vectorstore.similarity_search(query, k=k)
    prompt = f"""You are a knowledge base assistant.
Answer the question using ONLY the context below.
If the answer is not contained in the context, respond exactly with:
"I don't have that information in the uploaded documents."

Context:
{context}

Question:
{query}
"""
    return llm.invoke(prompt).content


print(answer_from_knowledge_base("What is RAG?"))

Retrieval‑Augmented Generation (RAG) is a technique where relevant documents are retrieved from a knowledge base and passed to a large language model as context, enabling the model to answer questions grounded in real data rather than relying solely on what it memorized during training.


In [10]:
print(answer_from_knowledge_base("What's the best way to cook pasta?"))

I don't have that information in the uploaded documents.


In [11]:
from langchain_core.tools import tool

@tool
def knowledge_base_search(query: str) -> str:
    """Search uploaded documents and answer using only that content."""
    return answer_from_knowledge_base(query)

In [15]:
print(knowledge_base_search.invoke("What is LangChain?"))

LangChain is a framework for building applications powered by large language models. It provides abstractions for chains, agents, memory, and retrieval, making it easier to combine LLMs with external tools and data sources.
